 #### Notebook 03: Random Forest Classifier
 #### Author: HKK Perera
 #### Inputs:  X_train/test.csv, y_train/test.csv
 #### Outputs: rf_model.pkl, rf_results.csv, plots

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix,
    classification_report
)

 **---------------------- 1. LOAD SPLITS ------------------------------**

In [5]:
X_train = pd.read_csv('../outputs/csv/X_train.csv')
X_test  = pd.read_csv('../outputs/csv/X_test.csv')
y_train = pd.read_csv('../outputs/csv/y_train.csv').squeeze()  # squeeze → Series
y_test  = pd.read_csv('../outputs/csv/y_test.csv').squeeze()

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

X_train: (2883, 6)  |  X_test: (721, 6)
Features: ['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'UniqueProducts', 'PurchaseSpanDays']


NOTE : Random Forest doesnt need feature scaling\
decision tree based models split on thresholds, not distances

**------------- 2. BASELINE CROSS-VALIDATION ---------------------------------**

In [7]:
randomForest_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# printing the model evaluation metric before going forward with hyperparameter tuning
cv_scores = cross_val_score(randomForest_base, X_train, y_train, cv=5, scoring='roc_auc')
print(f"\nBaseline CV ROC-AUC : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
#ROC AUC tells how well model separates customers who will return vs wont return closer to 1 is good


Baseline CV ROC-AUC: 0.7038 ± 0.0198


**---------------- 3. HYPERPARAMETER TUNING ----------------------------**

In [8]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [5, 10, 15, None],
    'min_samples_split':[2, 5, 10],
    'max_features':     ['sqrt', 'log2']
}

#total combinations : 4 x 4 x 3 x 2 = 72

In [9]:
grid_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest params:    {grid_search.best_params_}")
print(f"Best CV AUC:    {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_

Fitting 5 folds for each of 72 candidates, totalling 360 fits

Best params:    {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 300}
Best CV AUC:    0.7444
